# NonNULL Framework v1.0.1 — Sổ Tay Thực Nghiệm Colab (Tiếng Việt)

Huấn luyện trọn bộ **NonNULL v1.0.1** (Adapter LoRA + Bộ gác cổng Trạng thái Ẩn 2.1 MB) chỉ trong **~4–6 phút** trên 1 GPU Colab (`T4`/`L4`/`A100`), giải phẫu hộp trắng theo tầng và tái lập 3 bộ chuẩn quốc tế.

* **Kho mã nguồn:** https://github.com/GiangNhanNghia/NonNULL-Framework | **DOI:** `10.5281/zenodo.24491428`


## 1. Tải Mã Nguồn, Cài Thư Viện & Chạy 20 Bài Kiểm Thử Đơn Vị


In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/GiangNhanNghia/NonNULL-Framework.git"
REPO_DIR = "/content/NonNULL-Framework"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

def run_live(args):
    cmd = [sys.executable, "-u"] + args
    print("\n>>> RUNNING:", " ".join(cmd), flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed (exit={rc}): {' '.join(cmd)}")

run_live(["-m", "pip", "install", "-q", "-U", "bitsandbytes>=0.46.1"])
run_live(["-m", "pip", "install", "-q", "-r", "requirements.txt"])
run_live(["-m", "unittest", "discover", "-s", "tests", "-v"])


## 2. Tự Dò Tầng + Huấn Luyện SFT, Empathic LN-DPO & Bộ Gác Cổng 0% Rò Rỉ (~4–6 phút)


In [ ]:
MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # Or unsloth/Qwen2.5-7B-Instruct-bnb-4bit

# 1. Auto-probe optimal hidden layers (~15s)
run_live(["whitebox_probe.py", "--model_id", MODEL_ID, "--output", "./probe_v101.json", "--seed", "42"])

# 2. Full-Layer Hierarchical SFT (~2.5 min on L4)
run_live([
    "run_training.py", "--model_id", MODEL_ID,
    "--data_path", "./nonnull_train_100.jsonl",
    "--output_dir", "./output/euler_checkpoints",
    "--adapter_name", "adapter_v101_sft", "--epochs", "5", "--seed", "42", "--skip_test",
])

# 3. Empathic Length-Normalized DPO (~1.5 min on L4)
run_live([
    "run_dpo_v1.py", "--model_id", MODEL_ID,
    "--init_adapter", "./output/euler_checkpoints/adapter_v101_sft",
    "--output_adapter", "./output/euler_checkpoints/adapter_v101_lndpo",
    "--dpo_data", "./dpo_train_50.jsonl", "--val_data", "./dpo_val_10.jsonl",
    "--epochs", "2", "--seed", "42",
])

# 4. Train 2.1 MB Binary Hidden-State Gate with 0% benchmark leakage (~45s)
run_live([
    "train_hidden_gate.py", "--model_id", MODEL_ID,
    "--probe_report", "./probe_v101.json", "--tau_zero_min", "0.22",
    "--output_path", "./output/euler_checkpoints/hidden_gate_v101.pt", "--seed", "42",
])


## 3. Giải Phẫu Hộp Trắng Theo Tầng (~1–2 phút)


In [ ]:
# Optional: Run White-Box Layer Anatomy (~1–2 min)
run_live([
    "whitebox_anatomy.py", "--model_id", MODEL_ID,
    "--adapter_path", "./output/euler_checkpoints/adapter_v101_lndpo",
    "--out_json", "./whitebox_anatomy_v101.json",
    "--out_png", "./whitebox_anatomy_v101.png", "--seed", "42",
])


## 4. Tùy Chọn: Tái Lập Trọn Bộ 3 Chuẩn Quốc Tế (HarmBench 200, MT-Bench 160, XSTest 450)


In [ ]:
# Optional: Reproduce full international benchmarks (~3–4 hrs from scratch on L4; ~1–1.5 hrs on A100)
run_live([
    "eval_public_benchmarks.py", "--bench", "all",
    "--model_id", MODEL_ID,
    "--adapter_path", "./output/euler_checkpoints/adapter_v101_lndpo",
    "--gate_path", "./output/euler_checkpoints/hidden_gate_v101.pt",
    "--harmbench_samples", "200", "--run_harmbench_cls",
    "--mtbench_samples", "80", "--xstest_samples", "450",
    "--output_dir", "./benchmark_outputs_v101", "--seed", "42",
])
